In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
DATA_FOLDER = Path(".")

results_path = (
    DATA_FOLDER /
    "cross_dataset_validation_results.csv"
)

predictions_path = (
    DATA_FOLDER /
    "cross_dataset_predictions.csv"
)

if not results_path.exists():
    raise FileNotFoundError(
        f"File not found: {results_path}"
    )

if not predictions_path.exists():
    raise FileNotFoundError(
        f"File not found: {predictions_path}"
    )

model_results = pd.read_csv(results_path)
prediction_data = pd.read_csv(predictions_path)

print("Files loaded successfully!")
print("Model results shape:", model_results.shape)
print("Prediction data shape:", prediction_data.shape)

Files loaded successfully!
Model results shape: (12, 7)
Prediction data shape: (172983, 5)


In [3]:
required_result_columns = [
    "training_dataset",
    "testing_dataset",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc"
]

required_prediction_columns = [
    "training_dataset",
    "testing_dataset",
    "actual_fraud",
    "predicted_fraud",
    "fraud_probability"
]

missing_result_columns = [
    column
    for column in required_result_columns
    if column not in model_results.columns
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in prediction_data.columns
]

if missing_result_columns:
    raise ValueError(
        "Missing result columns: "
        + str(missing_result_columns)
    )

if missing_prediction_columns:
    raise ValueError(
        "Missing prediction columns: "
        + str(missing_prediction_columns)
    )

print("All required columns are available!")

All required columns are available!


In [4]:
numeric_result_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc"
]

for column in numeric_result_columns:
    model_results[column] = pd.to_numeric(
        model_results[column],
        errors="coerce"
    )

prediction_data["actual_fraud"] = pd.to_numeric(
    prediction_data["actual_fraud"],
    errors="coerce"
)

prediction_data["predicted_fraud"] = pd.to_numeric(
    prediction_data["predicted_fraud"],
    errors="coerce"
)

prediction_data["fraud_probability"] = pd.to_numeric(
    prediction_data["fraud_probability"],
    errors="coerce"
)

prediction_data = prediction_data.dropna(
    subset=[
        "actual_fraud",
        "predicted_fraud",
        "fraud_probability"
    ]
).copy()

prediction_data["actual_fraud"] = (
    prediction_data["actual_fraud"].astype(int)
)

prediction_data["predicted_fraud"] = (
    prediction_data["predicted_fraud"].astype(int)
)

print("Data cleaned successfully!")

Data cleaned successfully!


In [5]:
conditions = [
    (
        (prediction_data["actual_fraud"] == 1) &
        (prediction_data["predicted_fraud"] == 1)
    ),
    (
        (prediction_data["actual_fraud"] == 0) &
        (prediction_data["predicted_fraud"] == 0)
    ),
    (
        (prediction_data["actual_fraud"] == 0) &
        (prediction_data["predicted_fraud"] == 1)
    ),
    (
        (prediction_data["actual_fraud"] == 1) &
        (prediction_data["predicted_fraud"] == 0)
    )
]

error_labels = [
    "True Positive",
    "True Negative",
    "False Positive",
    "False Negative"
]

prediction_data["prediction_result"] = np.select(
    conditions,
    error_labels,
    default="Unknown"
)

prediction_data[
    [
        "actual_fraud",
        "predicted_fraud",
        "fraud_probability",
        "prediction_result"
    ]
].head()

,actual_fraud,predicted_fraud,fraud_probability,prediction_result
0,0,0,0.030242,True Negative
1,0,0,0.032906,True Negative
2,0,0,0.051219,True Negative
3,0,0,0.013115,True Negative
4,0,0,0.076336,True Negative


In [6]:
error_summary = (
    prediction_data
    .groupby(
        [
            "training_dataset",
            "testing_dataset",
            "prediction_result"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

required_error_types = [
    "True Positive",
    "True Negative",
    "False Positive",
    "False Negative"
]

for error_type in required_error_types:

    if error_type not in error_summary.columns:
        error_summary[error_type] = 0

error_summary = error_summary.rename(
    columns={
        "True Positive": "true_positive",
        "True Negative": "true_negative",
        "False Positive": "false_positive",
        "False Negative": "false_negative"
    }
)

error_summary

prediction_result,training_dataset,testing_dataset,false_negative,false_positive,true_negative,true_positive
0,Credit Card Fraud,Financial Fraud,185,38,14431,0
1,Credit Card Fraud,Fraud Detection,357,33,11350,1
2,Credit Card Fraud,Synthetic Fraud,333,38,19169,0
3,Financial Fraud,Credit Card Fraud,122,19,11585,0
4,Financial Fraud,Fraud Detection,358,17,11366,0
5,Financial Fraud,Synthetic Fraud,332,29,19178,1
6,Fraud Detection,Credit Card Fraud,119,88,11516,3
7,Fraud Detection,Financial Fraud,185,28,14441,0
8,Fraud Detection,Synthetic Fraud,327,142,19065,6
9,Synthetic Fraud,Credit Card Fraud,122,37,11567,0


In [7]:
error_summary["total_transactions"] = (
    error_summary["true_positive"]
    + error_summary["true_negative"]
    + error_summary["false_positive"]
    + error_summary["false_negative"]
)

actual_fraud_total = (
    error_summary["true_positive"]
    + error_summary["false_negative"]
)

actual_normal_total = (
    error_summary["true_negative"]
    + error_summary["false_positive"]
)

error_summary["false_negative_rate"] = np.where(
    actual_fraud_total > 0,
    (
        error_summary["false_negative"]
        / actual_fraud_total
    ),
    0
)

error_summary["false_positive_rate"] = np.where(
    actual_normal_total > 0,
    (
        error_summary["false_positive"]
        / actual_normal_total
    ),
    0
)

error_summary["overall_error_rate"] = np.where(
    error_summary["total_transactions"] > 0,
    (
        error_summary["false_positive"]
        + error_summary["false_negative"]
    ) / error_summary["total_transactions"],
    0
)

error_summary.round(4)

prediction_result,training_dataset,testing_dataset,false_negative,false_positive,true_negative,true_positive,total_transactions,false_negative_rate,false_positive_rate,overall_error_rate
0,Credit Card Fraud,Financial Fraud,185,38,14431,0,14654,1.0000,0.0026,0.0152
1,Credit Card Fraud,Fraud Detection,357,33,11350,1,11741,0.9972,0.0029,0.0332
2,Credit Card Fraud,Synthetic Fraud,333,38,19169,0,19540,1.0000,0.0020,0.0190
3,Financial Fraud,Credit Card Fraud,122,19,11585,0,11726,1.0000,0.0016,0.0120
4,Financial Fraud,Fraud Detection,358,17,11366,0,11741,1.0000,0.0015,0.0319
5,Financial Fraud,Synthetic Fraud,332,29,19178,1,19540,0.9970,0.0015,0.0185
6,Fraud Detection,Credit Card Fraud,119,88,11516,3,11726,0.9754,0.0076,0.0177
7,Fraud Detection,Financial Fraud,185,28,14441,0,14654,1.0000,0.0019,0.0145
8,Fraud Detection,Synthetic Fraud,327,142,19065,6,19540,0.9820,0.0074,0.0240
9,Synthetic Fraud,Credit Card Fraud,122,37,11567,0,11726,1.0000,0.0032,0.0136


In [8]:
false_negative_cases = prediction_data[
    prediction_data["prediction_result"]
    == "False Negative"
].copy()

false_negative_cases = false_negative_cases.sort_values(
    by="fraud_probability",
    ascending=False
)

print(
    "Total false negative cases:",
    len(false_negative_cases)
)

false_negative_cases.head(10)

Total false negative cases: 2975


,training_dataset,testing_dataset,actual_fraud,predicted_fraud,fraud_probability,prediction_result
157562,Synthetic Fraud,Financial Fraud,1,0,0.493118,False Negative
125480,Fraud Detection,Synthetic Fraud,1,0,0.490737,False Negative
170545,Synthetic Fraud,Fraud Detection,1,0,0.485124,False Negative
165115,Synthetic Fraud,Fraud Detection,1,0,0.484101,False Negative
167818,Synthetic Fraud,Fraud Detection,1,0,0.483702,False Negative
165814,Synthetic Fraud,Fraud Detection,1,0,0.481419,False Negative
100424,Fraud Detection,Credit Card Fraud,1,0,0.479097,False Negative
70649,Financial Fraud,Synthetic Fraud,1,0,0.478785,False Negative
172120,Synthetic Fraud,Fraud Detection,1,0,0.475059,False Negative
162757,Synthetic Fraud,Fraud Detection,1,0,0.472293,False Negative


In [9]:
false_positive_cases = prediction_data[
    prediction_data["prediction_result"]
    == "False Positive"
].copy()

false_positive_cases = false_positive_cases.sort_values(
    by="fraud_probability",
    ascending=False
)

print(
    "Total false positive cases:",
    len(false_positive_cases)
)

false_positive_cases.head(10)

Total false positive cases: 605


,training_dataset,testing_dataset,actual_fraud,predicted_fraud,fraud_probability,prediction_result
168408,Synthetic Fraud,Fraud Detection,0,1,0.744583,False Positive
100020,Fraud Detection,Credit Card Fraud,0,1,0.742712,False Positive
164756,Synthetic Fraud,Fraud Detection,0,1,0.718199,False Positive
121683,Fraud Detection,Synthetic Fraud,0,1,0.709978,False Positive
128935,Fraud Detection,Synthetic Fraud,0,1,0.707082,False Positive
86604,Financial Fraud,Synthetic Fraud,0,1,0.704346,False Positive
172487,Synthetic Fraud,Fraud Detection,0,1,0.697338,False Positive
143097,Synthetic Fraud,Credit Card Fraud,0,1,0.694405,False Positive
135950,Synthetic Fraud,Credit Card Fraud,0,1,0.694386,False Positive
128638,Fraud Detection,Synthetic Fraud,0,1,0.693922,False Positive


In [10]:
average_model_results = (
    model_results
    .groupby("training_dataset")
    .agg(
        average_accuracy=("accuracy", "mean"),
        average_precision=("precision", "mean"),
        average_recall=("recall", "mean"),
        average_f1=("f1_score", "mean"),
        average_roc_auc=("roc_auc", "mean")
    )
    .reset_index()
)

average_model_results.round(4)

,training_dataset,average_accuracy,average_precision,average_recall,average_f1,average_roc_auc
0,Credit Card Fraud,0.9775,0.0098,0.0009,0.0017,0.4274
1,Financial Fraud,0.9792,0.0111,0.0010,0.0018,0.4360
2,Fraud Detection,0.9813,0.0245,0.0142,0.0177,0.5614
3,Synthetic Fraud,0.9775,0.0573,0.0092,0.0149,0.5382


In [11]:
average_error_results = (
    error_summary
    .groupby("training_dataset")
    .agg(
        total_false_positives=(
            "false_positive",
            "sum"
        ),
        total_false_negatives=(
            "false_negative",
            "sum"
        ),
        average_false_positive_rate=(
            "false_positive_rate",
            "mean"
        ),
        average_false_negative_rate=(
            "false_negative_rate",
            "mean"
        ),
        average_error_rate=(
            "overall_error_rate",
            "mean"
        )
    )
    .reset_index()
)

average_error_results.round(4)

,training_dataset,total_false_positives,total_false_negatives,average_false_positive_rate,average_false_negative_rate,average_error_rate
0,Credit Card Fraud,109,875,0.0025,0.9991,0.0225
1,Financial Fraud,65,812,0.0015,0.9990,0.0208
2,Fraud Detection,258,631,0.0056,0.9858,0.0187
3,Synthetic Fraud,173,657,0.0050,0.9908,0.0225


In [12]:
final_comparison = pd.merge(
    average_model_results,
    average_error_results,
    on="training_dataset",
    how="left"
)

final_comparison.round(4)

,training_dataset,average_accuracy,average_precision,average_recall,average_f1,average_roc_auc,total_false_positives,total_false_negatives,average_false_positive_rate,average_false_negative_rate,average_error_rate
0,Credit Card Fraud,0.9775,0.0098,0.0009,0.0017,0.4274,109,875,0.0025,0.9991,0.0225
1,Financial Fraud,0.9792,0.0111,0.0010,0.0018,0.4360,65,812,0.0015,0.9990,0.0208
2,Fraud Detection,0.9813,0.0245,0.0142,0.0177,0.5614,258,631,0.0056,0.9858,0.0187
3,Synthetic Fraud,0.9775,0.0573,0.0092,0.0149,0.5382,173,657,0.0050,0.9908,0.0225


In [13]:
final_comparison["final_selection_score"] = (
    0.35 * final_comparison["average_recall"].fillna(0)
    + 0.25 * final_comparison["average_f1"].fillna(0)
    + 0.15 * final_comparison["average_precision"].fillna(0)
    + 0.15 * final_comparison["average_roc_auc"].fillna(0)
    + 0.10 * (
        1
        - final_comparison[
            "average_false_negative_rate"
        ].fillna(1)
    )
)

final_comparison = (
    final_comparison
    .sort_values(
        by="final_selection_score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_comparison["model_rank"] = (
    np.arange(1, len(final_comparison) + 1)
)

final_comparison[
    [
        "model_rank",
        "training_dataset",
        "average_precision",
        "average_recall",
        "average_f1",
        "average_roc_auc",
        "average_false_negative_rate",
        "final_selection_score"
    ]
].round(4)

,model_rank,training_dataset,average_precision,average_recall,average_f1,average_roc_auc,average_false_negative_rate,final_selection_score
0,1,Fraud Detection,0.0245,0.0142,0.0177,0.5614,0.9858,0.0987
1,2,Synthetic Fraud,0.0573,0.0092,0.0149,0.5382,0.9908,0.0972
2,3,Financial Fraud,0.0111,0.0010,0.0018,0.4360,0.9990,0.0680
3,4,Credit Card Fraud,0.0098,0.0009,0.0017,0.4274,0.9991,0.0664


In [14]:
if final_comparison.empty:
    raise ValueError(
        "No valid models are available for selection."
    )

best_model = final_comparison.iloc[0]

best_training_dataset = (
    best_model["training_dataset"]
)

print("FINAL MODEL SELECTION")
print("---------------------")
print("Algorithm: Random Forest")
print(
    "Training dataset:",
    best_training_dataset
)
print(
    "Average precision:",
    round(best_model["average_precision"], 4)
)
print(
    "Average recall:",
    round(best_model["average_recall"], 4)
)
print(
    "Average F1 score:",
    round(best_model["average_f1"], 4)
)
print(
    "Average ROC-AUC:",
    round(best_model["average_roc_auc"], 4)
)
print(
    "False negative rate:",
    round(
        best_model[
            "average_false_negative_rate"
        ],
        4
    )
)
print(
    "Final selection score:",
    round(
        best_model["final_selection_score"],
        4
    )
)

FINAL MODEL SELECTION
---------------------
Algorithm: Random Forest
Training dataset: Fraud Detection
Average precision: 0.0245
Average recall: 0.0142
Average F1 score: 0.0177
Average ROC-AUC: 0.5614
False negative rate: 0.9858
Final selection score: 0.0987


In [15]:
best_combination = (
    model_results
    .sort_values(
        by="f1_score",
        ascending=False
    )
    .iloc[0]
)

worst_combination = (
    model_results
    .sort_values(
        by="f1_score",
        ascending=True
    )
    .iloc[0]
)

print("BEST CROSS-DATASET COMBINATION")
print(
    "Train:",
    best_combination["training_dataset"]
)
print(
    "Test:",
    best_combination["testing_dataset"]
)
print(
    "F1 score:",
    round(best_combination["f1_score"], 4)
)

print("\nWORST CROSS-DATASET COMBINATION")
print(
    "Train:",
    worst_combination["training_dataset"]
)
print(
    "Test:",
    worst_combination["testing_dataset"]
)
print(
    "F1 score:",
    round(worst_combination["f1_score"], 4)
)

BEST CROSS-DATASET COMBINATION
Train: Fraud Detection
Test: Credit Card Fraud
F1 score: 0.0282

WORST CROSS-DATASET COMBINATION
Train: Credit Card Fraud
Test: Financial Fraud
F1 score: 0.0


In [16]:
plot_data = final_comparison.copy()

x_positions = np.arange(len(plot_data))
bar_width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(
    x_positions - bar_width / 2,
    plot_data["average_false_positive_rate"],
    width=bar_width,
    label="False Positive Rate",
    color="orange"
)

plt.bar(
    x_positions + bar_width / 2,
    plot_data["average_false_negative_rate"],
    width=bar_width,
    label="False Negative Rate",
    color="red"
)

plt.xticks(
    x_positions,
    plot_data["training_dataset"],
    rotation=20
)

plt.xlabel("Training Dataset")
plt.ylabel("Average Error Rate")
plt.title("Cross-Dataset Fraud Error Analysis")
plt.legend()
plt.grid(
    axis="y",
    alpha=0.3
)
plt.tight_layout()

plt.savefig(
    "fraud_error_analysis.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    "Chart saved as fraud_error_analysis.png"
)

Chart saved as fraud_error_analysis.png


In [17]:
final_model_selection = pd.DataFrame({
    "selected_algorithm": [
        "Random Forest"
    ],
    "best_cross_dataset_training_source": [
        best_training_dataset
    ],
    "deployment_training_dataset": [
        "train_fraud_balanced.csv"
    ],
    "selection_purpose": [
        "Cross-dataset generalization analysis"
    ],
    "average_precision": [
        best_model["average_precision"]
    ],
    "average_recall": [
        best_model["average_recall"]
    ],
    "average_f1": [
        best_model["average_f1"]
    ],
    "average_roc_auc": [
        best_model["average_roc_auc"]
    ],
    "average_false_positive_rate": [
        best_model[
            "average_false_positive_rate"
        ]
    ],
    "average_false_negative_rate": [
        best_model[
            "average_false_negative_rate"
        ]
    ],
    "final_selection_score": [
        best_model["final_selection_score"]
    ]
})

final_model_selection.round(4)

,selected_algorithm,best_cross_dataset_training_source,deployment_training_dataset,selection_purpose,average_precision,average_recall,average_f1,average_roc_auc,average_false_positive_rate,average_false_negative_rate,final_selection_score
0,Random Forest,Fraud Detection,train_fraud_balanced.csv,Cross-dataset generalization analysis,0.0245,0.0142,0.0177,0.5614,0.0056,0.9858,0.0987


In [18]:
prediction_data.to_csv(
    "fraud_prediction_error_details.csv",
    index=False
)

error_summary.to_csv(
    "fraud_error_summary.csv",
    index=False
)

final_comparison.to_csv(
    "final_model_comparison.csv",
    index=False
)

final_model_selection.to_csv(
    "final_model_selection.csv",
    index=False
)

false_negative_cases.to_csv(
    "false_negative_fraud_cases.csv",
    index=False
)

false_positive_cases.to_csv(
    "false_positive_fraud_cases.csv",
    index=False
)

print("Files saved successfully:")
print("1. fraud_prediction_error_details.csv")
print("2. fraud_error_summary.csv")
print("3. final_model_comparison.csv")
print("4. final_model_selection.csv")
print("5. false_negative_fraud_cases.csv")
print("6. false_positive_fraud_cases.csv")

Files saved successfully:
1. fraud_prediction_error_details.csv
2. fraud_error_summary.csv
3. final_model_comparison.csv
4. final_model_selection.csv
5. false_negative_fraud_cases.csv
6. false_positive_fraud_cases.csv
